# 🔗 Hands-On Bab 10 — Aplikasi AJS
### Analisis Jaringan Sosial: Konsep, Metode, dan Aplikasi

Notebook ini mensimulasikan penerapan AJS pada empat domain dari buku: **media sosial (deteksi KOL & echo chamber), kesehatan (super-spreader), bisnis (ONA/key connector),** dan **keamanan (deteksi pola jaringan mencurigakan)** — lengkap dengan **Latihan Soal Bab 10**.

In [1]:
!pip install -q networkx matplotlib
import networkx as nx
import matplotlib.pyplot as plt
import random

## 10.1 Media Sosial — Identifikasi Key Opinion Leader (KOL)

Bangun jaringan media sosial sederhana dan bandingkan tiga metrik (degree, eigenvector, betweenness) untuk menemukan tipe "pengaruh" yang berbeda, sesuai penjelasan Bab 10.1.2.

In [3]:
G_medsos = nx.watts_strogatz_graph(30, 4, p=0.1, seed=7)  # small-world graph, to show varied influence types

deg = nx.degree_centrality(G_medsos)
eig = nx.eigenvector_centrality(G_medsos, max_iter=1000)
bet = nx.betweenness_centrality(G_medsos)

top_degree = max(deg, key=deg.get)
top_eigen  = max(eig, key=eig.get)
top_bet    = max(bet, key=bet.get)

print(f"Simpul dengan degree tertinggi (paling banyak follower)      : {top_degree}")
print(f"Simpul dengan eigenvector tertinggi (KOL sesungguhnya)       : {top_eigen}")
print(f"Simpul dengan betweenness tertinggi (penjembatan antarkelompok): {top_bet}")

Simpul dengan degree tertinggi (paling banyak follower)      : 2
Simpul dengan eigenvector tertinggi (KOL sesungguhnya)       : 18
Simpul dengan betweenness tertinggi (penjembatan antarkelompok): 18


## 10.1.3 Deteksi Echo Chamber

Bangun dua komunitas dengan sedikit sisi penghubung, lalu ukur density internal vs. eksternal.

In [4]:
G_echo = nx.connected_caveman_graph(2, 8)  # 2 klik besar terhubung minimal

komunitas = nx.community.louvain_communities(G_echo, seed=1)
modularity = nx.community.modularity(G_echo, komunitas)

print(f"Jumlah komunitas terdeteksi: {len(komunitas)}")
print(f"Modularity: {modularity:.3f}  (modularity tinggi -> indikasi echo chamber kuat)")

for i, c in enumerate(komunitas):
    internal_edges = G_echo.subgraph(c).number_of_edges()
    print(f"  Komunitas {i}: {len(c)} anggota, {internal_edges} sisi internal")

Jumlah komunitas terdeteksi: 2
Modularity: 0.464  (modularity tinggi -> indikasi echo chamber kuat)
  Komunitas 0: 8 anggota, 27 sisi internal
  Komunitas 1: 8 anggota, 27 sisi internal


## 10.2 Kesehatan — Identifikasi Super-Spreader

Simulasikan jaringan kontak dan identifikasi individu berisiko tinggi menjadi super-spreader (degree sangat tinggi).

In [5]:
G_kontak = nx.barabasi_albert_graph(50, 3, seed=3)  # scale-free -> ada hub
degree_dict = dict(G_kontak.degree())
rata2_degree = sum(degree_dict.values()) / len(degree_dict)
super_spreaders = [n for n, d in degree_dict.items() if d > 3 * rata2_degree]

print(f"Rata-rata degree jaringan kontak: {rata2_degree:.1f}")
print(f"Kandidat super-spreader (degree > 3x rata-rata): {super_spreaders}")
print("\n(Pada dunia nyata, individu ini menjadi prioritas intervensi/isolasi lebih awal)")

Rata-rata degree jaringan kontak: 5.6
Kandidat super-spreader (degree > 3x rata-rata): [0, 4, 5]

(Pada dunia nyata, individu ini menjadi prioritas intervensi/isolasi lebih awal)


## 10.3 Bisnis — Organizational Network Analysis (ONA)

Simulasikan jaringan komunikasi internal organisasi dan identifikasi **key connector** serta potensi **silo**.

In [6]:
G_ona = nx.Graph()
tim_a = [f"A{i}" for i in range(1,5)]
tim_b = [f"B{i}" for i in range(1,5)]
tim_c = [f"C{i}" for i in range(1,5)]
for tim in [tim_a, tim_b, tim_c]:
    G_ona.add_edges_from([(tim[i], tim[j]) for i in range(len(tim)) for j in range(i+1, len(tim))])
# Hanya satu orang di tiap tim yang terhubung lintas tim (potensi bottleneck)
G_ona.add_edges_from([("A1","B1"), ("B1","C1")])

betweenness_ona = nx.betweenness_centrality(G_ona)
key_connector = max(betweenness_ona, key=betweenness_ona.get)
print(f"Key connector (betweenness tertinggi): {key_connector}")
print("--> Jika orang ini resign/cuti, komunikasi lintas tim berisiko terputus (bottleneck)")

isolasi = [n for n, d in G_ona.degree() if d <= 3]
print(f"\nKaryawan risiko isolasi (degree rendah, hanya terhubung dalam timnya sendiri): {isolasi}")

Key connector (betweenness tertinggi): B1
--> Jika orang ini resign/cuti, komunikasi lintas tim berisiko terputus (bottleneck)

Karyawan risiko isolasi (degree rendah, hanya terhubung dalam timnya sendiri): ['A2', 'A3', 'A4', 'B2', 'B3', 'B4', 'C2', 'C3', 'C4']


## ✅ Latihan Soal Bab 10

**1.** Jelaskan mengapa betweenness centrality menjadi metrik yang relevan di hampir semua domain aplikasi yang dibahas dalam bab ini (media sosial, kesehatan, bisnis, dan keamanan). Berikan contoh konkret untuk masing-masing domain!

**2.** Konsep "super-spreader" digunakan baik dalam konteks penyebaran informasi di media sosial maupun penyebaran penyakit dalam epidemiologi. Jelaskan persamaan dan perbedaan konsep ini di kedua domain tersebut.

### ✏️ Jawaban Soal 1–2 (tulis di sini)

_1._

_2._

### 🧮 Latihan tambahan
Gunakan salah satu jaringan simulasi di atas (medsos/kesehatan/bisnis), lalu bandingkan top-3 simpul di setiap metrik centrality untuk memperkuat jawaban Soal 1.

In [ ]:
G_pilihan = G_ona  # ganti dengan G_medsos atau G_kontak jika ingin domain lain

metrics = {
    "Degree": nx.degree_centrality(G_pilihan),
    "Betweenness": nx.betweenness_centrality(G_pilihan),
    "Closeness": nx.closeness_centrality(G_pilihan),
}
for nama, m in metrics.items():
    top3 = sorted(m.items(), key=lambda x: -x[1])[:3]
    print(f"{nama} -- top 3: {top3}")

---
### 📚 Referensi
Bab 10 — *Aplikasi AJS*, dalam **Analisis Jaringan Sosial: Konsep, Metode, dan Aplikasi**.

Lanjutkan ke **Notebook Bab 11 — Tantangan AJS** (bab penutup).